In [1]:
import rdkit
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem
import numpy as np
import pandas as pd
import QCflow
from QCflow.load_gaussian import *
from QCflow.torsion_parser import *
from QCflow.find_torsion import *
from QCflow.write_psi4 import *
from QCflow.run_psi4 import *
from QCflow.energy_calculations import *
from molecule_mutation import *
from calculation_status import *
from datetime import datetime
import random
import argparse

# Define default variables
options = {
    'run_num': {'default': 0},
    'run_size' : {'default': 50}
}

# Create a parser for the arguments that can be changed by the user
#parser = argparse.ArgumentParser()
#for arg, opts in options.items():
    #parser.add_argument(f'--{arg}', type=type(opts['default']), default=opts['default'])
#args = parser.parse_args()

# Store run_num as a string as it is mainly used for naming/retreaving files
#if hasattr(args, 'run_num') and args.run_num:
    #run_num_str = str(args.run_num)
#else:
    #run_num_str = str(options['run_num']['default'])

#run_size = args.run_size if hasattr(args, 'run_size') else options['run_size']['default']

run_size = 200
run_num_str = '2'

#Mutate the elite molecules
#Includes new fragments, new linkers and even a whole new molecules
#User inputted balance for what type of mutations occure and how many new molecules are wanted
#Idea mutation weighting:
# 25% are new molecules, 25% have 1 biofragment fragment change, 25% non-biofragment change, 25% linker change

#load the elite df
elite_df = pd.read_csv(f'dataframes/elite_run_{run_num_str}_df.csv')
#get th smi strings of the eilte_25
elite_smi = dict(zip(elite_df['Name'], elite_df['SMILES']))
#load all the molecules that have ever been ran in the GA
all_ran_molecules_dic = open_dictionary('ga_dic/total_molecules_ran.json')

all_ran_smi_canon = {k: Chem.CanonSmiles(v, useChiral=0) for k, v in all_ran_molecules_dic.items()}

#linker_dic
linker_dic = open_dictionary('ga_dic/linker_dic.json')
#bio_dic
bio_dic = open_dictionary('ga_dic/bio_dic.json')
#non_bio_dic
non_bio_dic = open_dictionary('ga_dic/non_bio_dic.json')

new_study_molecules = {}
for k, v in elite_smi.items():
    #options to mutate
    choices = ['new_bio', 'new_non_bio', 'new_linker']
    #which random one is chosen for this molecule
    selected_choice = random.choices(choices, k=1)[0]

    if selected_choice == 'new_bio':
        #find the biofragment and replace it with a new biofragment
        new_molecule = swap_one_fragment(v, bio_dic, non_bio_dic, 'bio')

        #there are less bio fragments so could run out of them
        #this is a fail safe to make sure it doesn't get stuck
        attempts = 0
        while Chem.CanonSmiles(new_molecule, useChiral=0) in all_ran_smi_canon.values() and attempts < 15:
            new_molecule = swap_one_fragment(v, bio_dic, non_bio_dic, 'bio')
            attempts += 1
            print(f'New Bio attempt {attempts}')
        
        if attempts == 15:
            new_molecule = swap_one_fragment(v, bio_dic, non_bio_dic, 'non_bio')
            print(f'New Bio attempt maxed out, using non bio fragment')

            attempts = 0
            while Chem.CanonSmiles(new_molecule, useChiral=0) in all_ran_smi_canon.values() and attempts < 15:
                attempts += 1
                print(f'New non bio attempt {attempts}')
            if attempts == 15:
                continue
                print(f'New Bio attempt maxed out, moving on')
        
        last_key, last_value = list(all_ran_smi_canon.items())[-1]
        #updates what the key will be by turning to int and then back to str
        make_num = int(last_key) + 1
        #creates the new name for the molecule
        make_num_str = str(make_num)

        #updates the new molecules list
        new_study_molecules[make_num_str] = new_molecule
        #updates the ran dictionary so no overlap occures
        all_ran_smi_canon[make_num_str] = new_molecule

    if selected_choice == 'new_non_bio':
        #do this
        #find the non-biofragment and replace it with a new non-biofragment
        new_molecule = swap_one_fragment(v, bio_dic, non_bio_dic, 'non_bio')
        attempts = 0
        while Chem.CanonSmiles(new_molecule, useChiral=0) in all_ran_smi_canon.values() and attempts < 15:
            new_molecule = swap_one_fragment(v, bio_dic, non_bio_dic, 'non_bio')
            attempts += 1
            print(f'New Bio attempt {attempts}')
        
        if attempts == 15:
            new_molecule = swap_one_fragment(v, bio_dic, non_bio_dic, 'bio')
            print(f'New non Bio attempt maxed out, using bio fragment')

            attempts = 0
            while Chem.CanonSmiles(new_molecule, useChiral=0) in all_ran_smi_canon.values() and attempts < 15:
                attempts += 1
                print(f'New bio attempt {attempts}')
            if attempts == 15:
                continue
                print(f'New non Bio attempt maxed out, moving on')
        
        last_key, last_value = list(all_ran_smi_canon.items())[-1]
        #updates what the key will be by turning to int and then back to str
        make_num = int(last_key) + 1
        #creates the new name for the molecule
        make_num_str = str(make_num)
        #updates the new molecules list
        new_study_molecules[make_num_str] = new_molecule
        #updates the ran dictionary so no overlap occures
        all_ran_smi_canon[make_num_str] = new_molecule

    if selected_choice == 'new_linker':
        #do this
        #find the linker and replace it with a new linker that is not the same as old one
        #swap the linker
        linker_type = find_linker_type(Chem.MolFromSmiles(v))
        #fragment the molecule
        fragments = fragment_molecule(Chem.MolFromSmiles(v), linker_type)
        #replace the linker
        replaced_linker = replace_linker(fragments, linker_dic)

        #make sure no duplicate linker occures
        #if the linker is tried lots of times then just replace the non_bio_fragment
        attempts = 0
        while Chem.CanonSmiles(replaced_linker, useChiral=0) in all_ran_smi_canon.values() and attempts < 15:
            replaced_linker = replace_linker(fragments, linker_dic)
            attempts += 1
            print(f'New linker attempt {attempts}')
        
        if attempts == 15:
            replaced_linker = swap_one_fragment(v, bio_dic, non_bio_dic, 'non_bio')
            print(f'New linker attempt maxed out, using non bio fragment')

            attempts = 0
            while Chem.CanonSmiles(replaced_linker, useChiral=0) in all_ran_smi_canon.values() and attempts < 15:
                replaced_linker = swap_one_fragment(v, bio_dic, non_bio_dic, 'non_bio')
                attempts += 1
                print(f'New bio attempt {attempts}')
            if attempts == 15:
                continue
                print(f'New non Bio attempt maxed out for linker, moving on')

        last_key, last_value = list(all_ran_smi_canon.items())[-1]
        #updates what the key will be by turning to int and then back to str
        make_num = int(last_key) + 1
        #creates the new name for the molecule
        make_num_str = str(make_num)
        #updates the new molecules list
        new_study_molecules[make_num_str] = replaced_linker
        #updates the ran dictionary so no overlap occures
        all_ran_smi_canon[make_num_str] = replaced_linker

#make a new list of molecules to run and make it same length as the molecule ran in list
current_run = open_dictionary(f'submission_dic/molecules_to_run_{run_num_str}.json')
length_of_run = len(current_run)
length_of_new_run = len(new_study_molecules)
#new_molecules_needed = length_of_run - length_of_new_run

#Makes sure the next run maintains a specific size so that the GA is healthy
new_molecules_needed = run_size - length_of_new_run

if new_molecules_needed > 0:
#new molecules to make up the numbers lost via elite step
    new_molecules = {}
    for i in range(new_molecules_needed + 1):
        
        #pick a random linker
        random_linker_type = random.choice(list(linker_dic.keys()))
        random_linker = linker_dic[random_linker_type]

        #pick a random bio fragment
        random_bio = random.choice(list(bio_dic.keys()))
        fragment_1 = bio_dic[random_bio]

        #pick at random if the second fragment will be bio or non_bio
        weights = [0.25, 0.75]  # 70% chance for option1, 30% chance for option2
        choices = ['bio', 'non_bio']
        selected_choice = random.choices(choices, weights=weights, k=1)[0]
        if selected_choice == 'bio':
            random_bio_2 = random.choice(list(bio_dic.keys()))
            fragment_2 = bio_dic[random_bio_2]

        if selected_choice == 'non_bio':
            random_non_bio = random.choice(list(non_bio_dic.keys()))
            fragment_2 = non_bio_dic[random_non_bio]

        #connect them all together
        frag_1_and_linker = combine_structure(fragment_1, random_linker)
        final_mol = combine_structure(frag_1_and_linker, fragment_2)

        attempts = 0
        #make sure its a molecule that hasn't been ran before
        while Chem.CanonSmiles(final_mol, useChiral=0) in all_ran_smi_canon.values() and attempts < 15:
            #pick a random linker
            attempts += 1
            print(f'New molecule {attempts}')
            random_linker_type = random.choice(list(linker_dic.keys()))
            random_linker = linker_dic[random_linker_type]

            #pick a random bio fragment
            random_bio = random.choice(list(bio_dic.keys()))
            fragment_1 = bio_dic[random_bio]

            #pick at random if the second fragment will be bio or non_bio
            weights = [0.25, 0.75]  # 25% chance for bio, 75% chance for non_bio
            choices = ['bio', 'non_bio']
            selected_choice = random.choices(choices, weights=weights, k=1)[0]
            if selected_choice == 'bio':
                random_bio_2 = random.choice(list(bio_dic.keys()))
                fragment_2 = bio_dic[random_bio_2]

            if selected_choice == 'non_bio':
                random_non_bio = random.choice(list(non_bio_dic.keys()))
                fragment_2 = non_bio_dic[random_non_bio]

            #connect them all together
            frag_1_and_linker = combine_structure(fragment_1, random_linker)
            final_mol = combine_structure(frag_1_and_linker, fragment_2)
        
        if attempts == 15:
            new_molecules_needed - 1
            print(f'New molecule maxed out')
            continue

        new_molecules[str(int(i))] = final_mol

else:
    new_molecules = {}


# Extract the last key from the reference dictionary
last_key = list(all_ran_molecules_dic.keys())[-1]

# Determine the base number from the last key
base_number = int(last_key)

# Renumber the new dictionary
new_molecules_renumbered = {f"{base_number + 1 + i}": v for i, (k, v) in enumerate(new_molecules.items())}

if len(new_molecules_renumbered) == 0:
    molecules_to_run = new_study_molecules

    run_num_int = int(run_num_str)
    new_run_num = run_num_int + 1
    new_run_num_str = str(new_run_num)

    save_dictionary(molecules_to_run, f'submission_dic/molecules_to_run_{new_run_num_str}.json')

else:

    # Extract the last key from the reference dictionary
    last_key = list(new_molecules_renumbered.keys())[-1]

    # Determine the base number from the last key
    base_number = int(last_key)

    # Renumber the new dictionary
    renumbered_dict_mutation = {f"{base_number + 1 + i}": v for i, (k, v) in enumerate(new_study_molecules.items())}

    molecules_to_run = new_molecules_renumbered | renumbered_dict_mutation

    run_num_int = int(run_num_str)
    new_run_num = run_num_int + 1
    new_run_num_str = str(new_run_num)

    save_dictionary(molecules_to_run, f'submission_dic/molecules_to_run_{new_run_num_str}.json')

#molecules_to_run = Chem.CanonSmiles(molecules_to_run, useChiral=0)

FileNotFoundError: [Errno 2] No such file or directory: 'dataframes/elite_run_2_df.csv'

In [6]:
len(molecules_to_run)

201

In [11]:
save_dictionary(molecules_to_run, f'submission_dic/molecules_to_run_{new_run_num_str}.json')

progress_file_path = 'GA_status.txt'
current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# Open the file in append mode and write some content
with open(progress_file_path, 'a') as file:
    file.write(f'elite_mutation complete for run {run_num_str} at {current_time}.\n')

In [9]:
molecules_to_run

{'357': 'COc1cc(-c2c(OC)c(OC)cc3[nH]c(C(=O)O)cc23)s:C:1c1c(C(F)(F)F)ccc2ccccc12',
 '358': 'COc1cc2cc(-c3cc(OC):C(c4cc(Br)cs4):s3)[nH]c2cc1OC',
 '359': 'COc1cc2c(ccn2C):C(c2cc(S(=O)(=O)O)cc3c2SCS3):c1OC',
 '360': 'COc1cc2cc[nH]c2c(C#Cc2nccc3nsnc23)c1OC',
 '361': 'COc1cc(-c2c(OC)c(OC)cc3cc[nH]c23)s:C:1c1c(OC)c(OC)cc2[nH]c(C(=O)O)cc12',
 '362': 'COc1cc2ccn(C)c2c(C=Cc2c(O)ncc3nonc23)c1OC',
 '363': 'COc1cc2cc(C(=O)O)[nH]c2c(C=Cc2c(OC)c(OC)cc3cc(C(=O)O)n(C)c23)c1OC',
 '364': 'COc1cc2cc[nH]c2c(C=Cc2nc(OC(C)=O)cc3nsnc23)c1OC',
 '365': 'COc1cc2cc(C#Cc3c(OC)c(OC)cc4ccn(C)c34)[nH]c2cc1OC',
 '366': 'COc1cc2cc(C3:c(OC)c(OC)cc4ccn(C)c:34)n(C)c2cc1OC',
 '367': 'COc1cc2[nH]c(C(=O)O)cc2c(C#Cc2c(OC)c(OC)cc3ccn(C)c23)c1OC',
 '368': 'COc1cc2[nH]ccc2c(C#Cc2c(OC)c(OC)cc3c2ccn3C)c1OC',
 '369': 'COc1cc2c(ccn2C)c(C#Cc2c(OC)c(OC)cc3cc[nH]c23)c1OC',
 '370': 'CNc1cnc2coc(C3:sc(-c4c(OC)c(OC)cc5c4ccn5C)cc:3OC)c2n1',
 '371': 'COc1cc2cc[nH]c2c(C=Nc2cccc3ncc(S(=O)(=O)O)nc23)c1OC',
 '372': 'COc1cc2cc(C(=O)O)n(C)c2c(C=C

In [10]:
molecules_to_run_test = Chem.CanonSmiles(molecules_to_run, useChiral=0)

TypeError: No registered converter was able to produce a C++ rvalue of type std::__cxx11::basic_string<wchar_t, std::char_traits<wchar_t>, std::allocator<wchar_t> > from this Python object of type dict

In [1]:
import rdkit
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem
import numpy as np
import pandas as pd
import QCflow
from QCflow.load_gaussian import *
from QCflow.torsion_parser import *
from QCflow.find_torsion import *
from QCflow.write_psi4 import *
from QCflow.run_psi4 import *
from QCflow.energy_calculations import *
from molecule_mutation import *
from calculation_status import *
from datetime import datetime
import random
import argparse

In [2]:
empty_dic = {}

In [4]:
save_dictionary(empty_dic, 'failed_dic/failed_D_A_match_0_molecules.json')
save_dictionary(empty_dic, 'failed_dic/failed_molecules_run_0.json')

In [5]:
pwd

'/cephfs/volumes/hpc_data_prj/ch_mime/fef711ab-2a13-4a43-a5d2-4b0d9b4300b6/GA_new_bio_16_06_25/archive_testing_attempt_5/MithrilMolGA/MithrilMolGA'